<html>
    <div style="margin: 0px 20px;">
        <h1 style="color: #045A7A; text-align: center;">One Codex Notebooks – Tutorial</h1>
        <p>In this tutorial, we'll highlight how to fetch samples and analysis results, access data from your projects, collate results across many samples from a study, get started with basic exploratory plotting, and calculate several common diversity statistics from the literature.</p>
        <p>To follow along, log into your account and click <b>"Copy this notebook"</b> on the top right. You can then make edits and experiment as you work your way through the tutorial.</p>
        <h5>A few quick tips to get you started...</h5>
        <p>
            We've built a library for working with our platform more easily from Python. It's quite simple, and you can get started quickly here with an example like the following: 
        </p>
        <pre>
from onecodex import Api
ocx = Api()  <em style="color: #888;"># This automatically connects to your account</em>
public_samples = ocx.Samples.where(public=True)

sample = ocx.Samples.get('e94934526edd49f6')  <em style="color: #888;"># Fetch a specific sample...</em>
sample.metadata                               <em style="color: #888;"># ...view its metadata...</em>
sample.metadata.name = 'My new name!'         <em style="color: #888;"># ...update...</em>
sample.metadata.save()                        <em style="color: #888;"># ...and save!</em></pre>
        <p>
            A few other resources you may find useful include: notes on getting started with <a href="https://onecodex.github.io/onecodex/"> our One Codex library</a>; the <a href="https://docs.onecodex.com">full documentation on our API</a> (more technical); a cheat sheet on <a href="https://nbviewer.org/github/pybokeh/jupyter_notebooks/blob/main/pandas/PandasCheatSheet.ipynb">getting started with Pandas</a>, a Python library for data manipulation; and <a href="http://blog.onecodex.com">reading a few of our blog posts</a> (where we use these notebooks extensively). As always, also feel free to send us questions or suggestions by clicking the chat icon in the bottom right!
        </p>
        <p style="color: #888; font-size: 0.95em;"><em>
            Ps. We'll include a similar header on all new notebooks as a quick getting started guide. Feel free to double click on this cell to edit and delete these tips.</em>
        </p>
    </div>
</html>

## 1. Create a One Codex API connection
This first step connects your notebook to the One Codex API, providing secure access to your private data as well as publicly shared samples available on our platform. By convention, we recommend instantiating a connection to the API with `ocx = Api()`.

In [ ]:
from onecodex import Api

ocx = Api()  # This automatically logs you into your account by  
             # injecting a JWT authentication token only valid for 24 hours 
             # (this notebook session)

## 2. Samples

### Get some public samples

In [ ]:
samples = ocx.Samples.where(public=True, limit=10)

In [ ]:
samples[0]  # Printing an API resource renders an HTML table

### Fetch a sample by filename

In [ ]:
# Find a public sample by filename
ocx.Samples.where(filename='EcoliAMR2.fastq.gz', public=True)

In [ ]:
# Find a sample in *your* account by filename
# Note: you will need to replace this with the filename for a sample in your account
ocx.Samples.where(filename="placeholder")

### Find a sample by its ID

In [ ]:
ocx.Samples.get('29c0503f12714d9d')

## 3. Jobs and Analyses

Each sample (FASTA/FASTQ) uploaded to One Codex is analyzed with at least one job. The current classification database is always run on each sample. Additional jobs can be run on your samples, including different job types.

An analysis can be thought of as a result of a job run on a sample.

### Get an Analysis

We're fetching a metagenomic classification result by its ID. You can also get a sample's classification result with `samples[0].primary_classification`. You can find a classification's unique ID in the URL when viewing the results page. Here, we'll use one of our public examples.

In [ ]:
analysis = ocx.Classifications.get('e18dc42b4f0d427b')

In [ ]:
analysis

### View Classification Results

In [ ]:
# Fetches the results and returns a Pandas DataFrame
df = analysis.table() 
df.head()

**Note**: Readcounts, Readcounts w/ Children, Abundance, and Abundance w/ Children
are some of the metrics we generate for classification results.
You can read more here:
https://docs.onecodex.com/en/articles/10245796-q-what-is-the-difference-between-readcount-readcount-with-children-and-abundances

In [ ]:
# Filter results to genus-level hits only
df[df['rank'] == 'genus'].head()  

### View details about the job

This is the strictly-versioned job run for this analysis
(code, execution environment, and reference database).
Note also the job's `analysis_type` here, which is `'classification'`--meaning that it produces taxonomic classifications for a sample. Other jobs can produce other types of results.

In [ ]:
analysis.job

### Find jobs by type

You can find jobs based on their `analysis_type`. These include: `classification`, `panel`, `alignment`, `control`, `report`, `functional`, `mlst`, and `workflow`.

In [ ]:
# For example, this is one of the user-runnable marker panels you can now find under
# the Run Workflows page -- an AMR panel:
ocx.Jobs.where(public=True, analysis_type='panel')[0]

## 4. Fetching samples from a project

[Projects](https://docs.onecodex.com/en/articles/3764345-sharing-projects) offer a useful mechanism for sharing and organizing your samples. In this tutorial, we're going to work with a public project consisting of data from the Sequence Read Archive (SRA). Since this data is public, the following example code should work from _any_ account.

In [ ]:
# Fetch a project by its ID.
#
# You can also fetch your own projects by the `name` field using a `.where()` 
# query. For example:
# 
# ocx.Projects.where(name="Infant Gut Microbiome, Karelia (All)")
#
# Alternatively, you can fetch a public project using its short `project_name`
# and by passing `public=True`. For example:
#
# ocx.Projects.where(public=True, project_name='prjna290380')
#

project = ocx.Projects.get('d53ad03b010542e3')

#### In this tutorial, we're going to work with the [DIABIMMUNE](https://pubs.broadinstitute.org/diabimmune/three-country-cohort) three country cohort:

Vatanen T, Kostic AD, d’Hennezel E, et al.<br>
[Variation in Microbiome LPS Immunogenicity Contributes to Autoimmunity in  Humans](https://www.ncbi.nlm.nih.gov/pubmed/27133167)<br>
_Cell_. 2016 May 5;165(4):842-53.

_Tip: You can inspect objects by simply entering them as a standalone notebook cell. Most objects should render as nice HTML tables displaying all their key properties._

In [ ]:
project

### Fetch this project's samples

In [ ]:
# This returns 0 samples because by default ocx.Samples.where
# searches samples in your account only
len(ocx.Samples.where(project=project.id))

In [ ]:
# So, we need to pass the `public=True` flag to ocx.Samples.where() in order to 
# search all public samples. Here, we limit this to return 50 samples. By default,
# queries return all private samples or up to 1000 public samples. Pass `limit=n`
# to retrieve up to `n` available public samples.
samples = ocx.Samples.where(project=project.id, public=True, limit=50)  

In [ ]:
len(samples)

### Now, let's analyze the results for the samples in this project

_Note: These samples have a *lot* of custom metadata. You can add custom metadata to your samples on our website, by uploading an Excel sample sheet, or via our API. For more details, see our documentation on metadata: [https://docs.onecodex.com/docs/metadata](https://docs.onecodex.com/en/articles/3764349-metadata)._

In [ ]:
# take a peek at what metadata is available for these samples:
samples[0].metadata

In [ ]:
# To keep things speedy, we're going to work with a subset of the samples from this project
ten_samples = samples[:10]

## 5. Heatmaps

Our library includes support for plotting heatmaps. By default, heatmaps display the top 10 taxa, but this can be changed by passing `top_n=`. Alternatively, pass `threshold=cutoff` to display only taxa above a certain abundance threshold, or use a combination of both `threshold` and `top_n`. Pass, e.g., `rank='genus'` to restrict the analysis to a specific taxonomic rank. You can customize the plot using `title=`. In addition, specify `xlabel=` or `ylabel=` to label the axes.

In [ ]:
ten_samples.plot_heatmap(
    metric="abundance_w_children",
    rank='genus', # try changing the rank to species
    title='Top 20 Genera',
    top_n=20
)

## 6. Stacked Bar Plots

Bar plots are a common way to view taxonomic abundance data


In [ ]:
ten_samples.plot_bargraph(
    rank='genus',
    legend='Genus',
    ylabel='Relative Abundance',
    top_n=25
)

### Barplot: single sample

Here, we're going to filter the samples to a single subject of the study and generate a stacked bar plot.


In [ ]:
# filter to samples from subject P014839
subject_samples = samples.filter(lambda s: s.metadata.custom["host_subject_id"] == "P014839")

# note: subject_samples is a collection containing a single sample
subject_samples.plot_bargraph(
    rank="genus",
    metric="normalized_readcount_w_children",
    label="host_age",
    title="Subject P014839",
    xlabel="Host Age at Sampling Time (days)",
    ylabel="Relative Abundance",
    legend="Genus",
)

## 7. Alpha diversity statistics
The `onecodex` Python client library provides rich access to the API layer of the One Codex platform. This includes several built-in functions for computing common alpha and beta diversity statistics. More statistics can be calculated by passing data from One Codex to the [`skbio`](https://scikit.bio/) library (the basis for much of QIIME2). 

First, we'll get started by computing [Simpson's index](https://scikit.bio/docs/latest/generated/skbio.diversity.alpha.simpson.html) for our subset of ten samples.

In [ ]:
ten_samples.alpha_diversity(diversity_metric='simpson')

Several other alpha (within-sample) diversity statistics are also included, e.g., [Shannon's diversity index](https://scikit.bio/docs/latest/generated/skbio.diversity.alpha.shannon.html). The results are returned as a pandas DataFrame, which can be manipulated to include sample names or other metadata.

**Note:** Shannon diversity is calculated using log base `e` (natural log).

In [ ]:
shannon = ten_samples.alpha_diversity(diversity_metric='shannon')
shannon['filename'] = [s.filename for s in ten_samples]
shannon['host_age_years'] = ten_samples.metadata['host_age'][shannon.index] / 365
shannon

## 8. Beta diversity statistics
Next, we'll explore a range of built-in beta diversity statistics. Most of these functions directly return a `skbio` [distance matrix object](https://scikit.bio/docs/latest/generated/skbio.stats.distance.DistanceMatrix.html).

In [ ]:
distance_matrix = ten_samples.beta_diversity(diversity_metric='braycurtis')

In [ ]:
# Printing shows the distance matrix. Here, the labels are classification IDs.
distance_matrix.to_data_frame()

**Note:** Alternatively, we can use the built-in `plot_distance` function to visualize a distance
matrix. The title of these plots can be set by passing the `title=` keyword argument.

In [ ]:
samples[:20].plot_distance(
    title='Bray Curtis Distance Matrix',
    # try some other diversity_metrics like `weighted_unifrac`
    diversity_metric='braycurtis'
)

**By the way**: You can always call `help()` on any of the functions in our library for a detailed description of what they do, and a few examples!

In [ ]:
help(samples.plot_distance)

## 9. Ordination

Our library supports ordination via Principal Component Analysis (PCA) and multidimensional scaling (MDS, more commonly known as PCoA). Ordination is useful for exploring the similarity of large numbers of samples in a single plot. Points on these plots can be colored and/or sized by taxon abundance and metadata.

These plots are fully interactive--try zooming in on specific regions, or hovering over points to see the abundance of Firmicutes taxa in each sample!

In [ ]:
samples.plot_pca(color='geo_loc_name', size='Bacteroides', tooltip='Firmicutes')

In [ ]:
# Specifying method='pcoa' uses classic PCoA via eigenvector decomposition
samples.plot_mds(
    diversity_metric='weighted_unifrac',
    method='pcoa',
    color='host_age',
    size='Bacteroides',
    tooltip='Firmicutes'
)

## 10. Plotting by metadata

Our library can generate quick exploratory plots with arbitrary metadata, using both built-in and custom fields. Specify the data to plot on the horizontal or vertical axes using `haxis` and `vaxis`, respectively. The vertical axis can be numerical metadata fields, taxon IDs or names (at any rank), or a diversity statistic. The horizontal axis can be one or more categorical metadata fields.

In [ ]:
cat = samples.plot_metadata(vaxis='Bacteroides', haxis='allergy_cat', return_chart=True)
dog = samples.plot_metadata(vaxis='Bacteroides', haxis='allergy_dog', return_chart=True)
birch = samples.plot_metadata(vaxis='Bacteroides', haxis='allergy_birch', return_chart=True)

# Use `return_chart=True` and display multiple plots side-by-side
cat | dog | birch

In [ ]:
catdog = samples.plot_metadata(
    vaxis='Bacteroidetes', haxis=('allergy_cat', 'allergy_dog'), return_chart=True
)
totalige = samples.plot_metadata(
    vaxis='totalige', haxis='Bacteroides', return_chart=True, ylabel='Total IgE'
)

catdog | totalige

Here, we display observed taxa, Simpson’s Index, and Shannon Entropy side-by-side, grouped by the region of birth.

In [ ]:
observed_taxa = samples.plot_metadata(vaxis="observed_taxa", haxis="geo_loc_name", return_chart=True)
simpson = samples.plot_metadata(vaxis="simpson", haxis="geo_loc_name", return_chart=True)
shannon = samples.plot_metadata(vaxis="shannon", haxis="geo_loc_name", return_chart=True)

observed_taxa | simpson | shannon

In [ ]:
cat_simpson = samples.plot_metadata(
    haxis='host_age', vaxis='simpson',
    title="Simpson's Index by Host Age",
    xlabel='Host Age', ylabel="Simpson's Index", return_chart=True
)
delivery_simpson = samples.plot_metadata(
    haxis='delivery', vaxis='simpson',
    title="Simpson's Index by Delivery Method",
    xlabel='Delivery Method', ylabel="Simpson's Index", return_chart=True
)

cat_simpson | delivery_simpson

The `plot_metadata` function can search through all taxa in your samples and pull out read counts or relative abundances. Here, we look at how the abundance of Bacteroides changes with host age.

In [ ]:
samples.plot_metadata(haxis="host_age", vaxis="Bacteroides", plot_type="scatter")

## 11. Advanced manipulation of results
Results from our platform can easily be accessed as pandas DataFrames. This makes it easy to perform any sort of manipulation you choose on the raw data. In the following example, we transform the data by grabbing results at the genus level, adding a pseudocount of 1, normalizing to 1.0, and taking the fourth root.

In [ ]:
# Grab genus-level results and don't remove taxa with zero reads
results = ten_samples.to_df(remove_zeros=False, rank='genus')
# Add a pseudocount of 1
results = results + 1
# Normalize to 1.0 in each sample
results = results.div(results.sum(axis=1), axis=0)
# Take the fourth root
results = results ** 0.25

results.head()

<html><h2 style="text-align: center">That's it! Please get in touch at support@onecodex.com if you have more questions or chat with us by opening the window on the bottom right. Thanks!</h2></html>